# 004 - Gold Load Revenue

This notebook aggregates silver-layer data into the **gold** layer, producing revenue summaries for analysis.

**Outputs:**
* `sandbox_general.gold.airbnb_per_listing` — total revenue per Airbnb listing
* `sandbox_general.gold.airbnb_per_postalcode` — total revenue per postal code (Airbnb)
* `sandbox_general.gold.rental_per_listing` — total revenue per rental listing
* `sandbox_general.gold.rental_per_postalcode` — total revenue per postal code (rentals)

**Sources:**
* `sandbox_general.silver.airbnb`
* `sandbox_general.silver.rentals`

Each gold table includes audit columns (`insertDateTime`, `updatedDateTime`) and is written in overwrite mode.

In [0]:
from pyspark.sql import functions as F

#set all variables to be used in this notebook
silver_rentals_table_name = f"sandbox_general.silver.rentals"
silver_airbnb_table_name = f"sandbox_general.silver.airbnb"
gold_airbnb_per_listing_table_name = f"sandbox_general.gold.airbnb_per_listing"
gold_airbnb_per_postalcode_table_name = f"sandbox_general.gold.airbnb_per_postalcode"
gold_rental_per_listing_table_name = f"sandbox_general.gold.rental_per_listing"
gold_rental_per_postalcode_table_name = f"sandbox_general.gold.rental_per_postalcode"

df_rental = spark.read.format("delta").table(silver_rentals_table_name)
df_airbnb = spark.read.format("delta").table(silver_airbnb_table_name)

In [0]:
df_airbnb_revenue_per_listing = (
     df_airbnb
    .groupBy("listingPk", "source", "latitude", "longitude")
    .agg(F.sum("price").alias("Revenue"))
    .select(
        "listingPk",
        "Revenue",
        F.col("source").alias("Source"),
        F.col("latitude").alias("Latitude"),
        F.col("longitude").alias("Longitude"),
        F.current_timestamp().alias("InsertedDateTime"),
        F.lit(None).cast("timestamp").alias("UpdatedDateTime"),
    )
)
#display(df_airbnb_revenue_per_listing)

listingPk,Revenue,Source,Latitude,Longitude,InsertedDateTime,UpdatedDateTime
33316f01d7997de2fe45aca9c83106a5,125,airbnb,52.34595819,4.921402399,2026-07-08T15:10:30.418Z,null
ebdd2a430790894111a30e2a722dc40d,75,airbnb,52.36982437,4.8523785,2026-07-08T15:10:30.418Z,null
624dbef3ec82bcaa7a8d1b7bfa5dc0c5,250,airbnb,52.40856537,4.907634099,2026-07-08T15:10:30.418Z,null
3e583fa071a49036590e68649e624476,90,airbnb,52.37010777,4.911900805,2026-07-08T15:10:30.418Z,null
c5be0fdce2b4a65e547cf72411dea33b,175,airbnb,52.37052357,4.856674898,2026-07-08T15:10:30.418Z,null
793031b9a03e8661a94d42ca91e91563,149,airbnb,52.33001301,4.856636145,2026-07-08T15:10:30.418Z,null
16b5f51703ac6f0d90f8a5a14caa097b,150,airbnb,52.37799865,4.882208965,2026-07-08T15:10:30.418Z,null
67f3e90a1cd81fa2bd87caf1e74b20dd,200,airbnb,52.36649743,4.89472163,2026-07-08T15:10:30.418Z,null
67e22c546fa1013d93a446d39d071adc,110,airbnb,52.37752227,4.85146996,2026-07-08T15:10:30.418Z,null
665a3636004a51fd1c484d7c08b87b24,119,airbnb,52.32450327,4.885269763,2026-07-08T15:10:30.418Z,null


In [0]:
df_airbnb_postalcode_revenue = (
    df_airbnb
    .groupBy("postalCode", "source")
    .agg(F.sum("price").alias("Revenue"))
    .select(
        F.col("postalCode").alias("PostalCode"),
        "Revenue",
        F.col("source").alias("Source"),
        F.current_timestamp().alias("InsertedDateTime"),
        F.lit(None).cast("timestamp").alias("UpdatedDateTime"),
    )
)
#display(df_airbnb_postalcode_revenue)

In [0]:
df_rental_revenue_per_listing = (
    df_rental
    .groupBy("listingPk", "source")
    .agg(F.sum("rentAmount").alias("Revenue"))
    .select(
        "listingPk",
        "Revenue",
        F.col("source").alias("Source"),
        F.current_timestamp().alias("InsertedDateTime"),
        F.lit(None).cast("timestamp").alias("UpdatedDateTime"),
    )
)

In [0]:
df_rental_revenue_per_postalcode = (df_rental.groupBy("postalCode", "source")
                .agg(F.sum("rentAmount").alias("Revenue"))
                .select(
                    F.col("postalCode").alias("PostalCode"),
                    "Revenue",
                    F.col("source").alias("Source"),
                    F.current_timestamp().alias("InsertedDateTime"),
                    F.lit(None).cast("timestamp").alias("UpdatedDateTime"),)
     )

In [0]:
df_airbnb_revenue_per_listing.write.mode("overwrite").saveAsTable(gold_airbnb_per_listing_table_name)
df_airbnb_postalcode_revenue.write.mode("overwrite").saveAsTable(gold_airbnb_per_postalcode_table_name)
df_rental_revenue_per_listing.write.mode("overwrite").saveAsTable(gold_rental_per_listing_table_name)
df_rental_revenue_per_postalcode.write.mode("overwrite").saveAsTable(gold_rental_per_postalcode_table_name)